In [ ]:
# ADD SENSOR TEST DATE-TIME

import os
import ROOT as root
import numpy as np
from array import array
import glob

txt_files = glob.glob("FBKdata/2x2/*.txt")

lines = []

for txt in txt_files:
    #print(txt)
    header = []

    with open(txt, 'r') as f:
        lines = f.readlines()

    with open(txt, 'w') as f:
        for m in txt:
            if m.isdigit():
                header.append(m)
        for number, line in enumerate(lines):
            if number == 0:
                f.write('producer structure wafer type grn grt column row \n')
            if number == 1:
                if int(header[3]) == 9:
                    f.write('FBK 16x16 '+str(header[2])+' '+str(header[3])+' '+str(header[4])+' '+str(header[5])+' '+str(header[6])+' '+str(header[7])+'\n')
                if int(header[3]) == 1:
                    f.write('FBK 16x16 '+str(header[2])+' '+str(header[3])+str(header[4])+' '+str(header[5])+' '+str(header[6])+' '+str(header[7])+' '+str(header[8])+'\n')
            if number >112:
                f.write(line)

In [ ]:
import os
import ROOT as root
import numpy as np
from array import array
import glob

file = root.TFile("/Users/icosivi/cernbox/MTD/QAQC/root_files/UFSDLF_1x1_CVtree_ETL-site.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
#type = array('i', [0])
#side = array('i', [0])
#grt = array('i', [0])
row = array('i', [0])
column = array('i', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()
#CP = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("type", type, 'type/I')
tree.Branch("side", side, 'side/I')
#tree.Branch("grt", grt, 'grt/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')


V.reserve(1000)
IBACK.reserve(1000)
#CP.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)
#tree.Branch("CPAD", "std::vector<float>", CP)

txt_files = glob.glob("/Users/icosivi/cernbox/MTD/QAQC/FBKdata/UFSD_LF/IV-CV_single-pad_W1_etl-site/W1/IV/*.txt")
#txt_files = glob.glob("/Users/icosivi/cernbox/MTD/QAQC/FBKdata/UFSD_LF/IV-CV_single-pad_W1_etl-site/W1/CV/*.txt")

counter = 0

for txt in txt_files:

    V.clear()
    IBACK.clear()
    #CP.clear()

    event[0] = counter

    with open(txt,"r") as t:
        #print(t)
        lines_list = t.readlines()
        header = lines_list[1]
        wafer[0] = int( header.split(" ")[2] )
        type[0] = int(1)
        print(str(header))
        if str(header.split(" ")[3])=='A':
         #print('pippo A')
         side[0] = int(0)
        elif str(header.split(" ")[3])=='B':
         #print('pippo B')
         side[0] = int(1)
        column[0] = int( header.split(" ")[4] )
        row[0] = int( header.split(" ")[5] )
        lines = lines_list[3:]

        '''
        for line in lines:
            if float(line.split("	")[1])<0:
                IBACK.push_back( -1*float(line.split("	")[1]) )
            else:
                IBACK.push_back( float(line.split(" ")[1]) )

            if(V.size()>3):
                if( -1*float(line.split("	")[0])>V.at( V.size()-1 ) ):
                    V.push_back( -1*float(line.split("	")[0]) )
                else:
                    break
            else:
                V.push_back( -1*float(line.split("	")[0]) )

        '''
        for line in lines:
            if float(line.split("	")[0])<0:
                if( V.size()>3):
                    if( -1*float(line.split("	")[0])>V.at( V.size()-1 ) ):
                        V.push_back( -1*float(line.split("	")[0]) )
                    else:
                        break
                else:
                    V.push_back( -1*float(line.split("	")[0]) )
            else:
                if( V.size()>3):
                    if( float(line.split("	")[0])>V.at( V.size()-1 ) ):
                        V.push_back( float(line.split("	")[0]) )
                    else:
                        break
                else:
                    V.push_back( float(line.split("	")[0]) )

            if float(line.split("	")[1])<0:
                #CP.push_back( -1*float(line.split("	")[1]) )
                IBACK.push_back( -1*float(line.split("	")[1]) )
            else:
                #CP.push_back( float(line.split("	")[1]) )
                IBACK.push_back( float(line.split("	")[1]) )

    tree.Fill()
    counter += 1

tree.Write()
file.Write()
file.Close()